In [1]:
# Zenodo → Google Drive SMART TRANSFER (resumable + status check)
from google.colab import drive
import requests, urllib3, os, time

# --- SETTINGS ---
zenodo_id = "14979685"
target_dir = "/content/drive/MyDrive/SLAM_Lite/zenodo_14979685"
threads = 8                 # parallel connections
segment_size = "10M"        # chunk size per connection
# ----------------

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Install aria2
!apt -qq install -y aria2

# 3. Create destination folder
os.makedirs(target_dir, exist_ok=True)

# 4. Fetch Zenodo file list
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
api_url = f"https://zenodo.org/api/records/{zenodo_id}"
print(f"Fetching metadata from {api_url} ...")
r = requests.get(api_url, verify=False, timeout=30)
r.raise_for_status()
data = r.json()

# 5. Build link list and check what is already downloaded
link_file = "/content/zenodo_links.txt"
existing_files = os.listdir(target_dir)
to_download = []

with open(link_file, "w") as f:
    for file in data.get("files", []):
        name = file["key"]
        url = file["links"]["self"] + "?download=1"
        if name in existing_files:
            size = os.path.getsize(os.path.join(target_dir, name)) / 1e6
            print(f"Already present: {name} ({size:.2f} MB)")
        else:
            to_download.append(url)
            f.write(url + "\n")

print(f"\n{len(existing_files)} files already present in Drive.")
print(f"{len(to_download)} files remaining to download.\n")

# 6. Start resumable download
if len(to_download) > 0:
    print("Starting resumable download in 5 seconds...")
    time.sleep(5)
    start = time.time()
    !aria2c -i /content/zenodo_links.txt -d "$target_dir" \
        -x {threads} -s {threads} -k {segment_size} --continue=true
    end = time.time()
    print(f"\nTransfer complete (elapsed {((end-start)/60):.1f} minutes)")
else:
    print("All files already downloaded.")

Mounted at /content/drive
The following additional packages will be installed:
  libaria2-0 libc-ares2
The following NEW packages will be installed:
  aria2 libaria2-0 libc-ares2
0 upgraded, 3 newly installed, 0 to remove and 41 not upgraded.
Need to get 1,513 kB of archives.
After this operation, 5,441 kB of additional disk space will be used.
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 121703 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Se